In [1]:
import pandas as pd
import joblib
import numpy as np
import threading

In [2]:
data = pd.read_csv("../new_code/DATASET.csv")
data2 = pd.DataFrame(None, columns = ['pred1','pred2','pred3','pred4','pred5','pred6','pred7','pred8','pred9','pred10','MEAN'])

In [3]:
model1 = joblib.load("/Users/anshumaansoni/PycharmProjects/battery-life-predictor/models/battery_random_forest_model.joblib")
model2 = joblib.load("/Users/anshumaansoni/PycharmProjects/battery-life-predictor/models/battery_random_forest_model2.joblib")

In [4]:
TIME_STEPS = 10
SIM_INTERVAL_MINUTES = 5.0
SIM_INTERVAL_HOURS = SIM_INTERVAL_MINUTES / 60.0
DEGRADATION_RATE_PER_INTERVAL = 0.000005
MAX_OPERATIONAL_VOLTAGE = 12.6
MIN_OPERATIONAL_VOLTAGE = 9.4
MAX_SIMULATED_CURRENT = 15.0

In [5]:
battery_type_capacities = {
    "B1":81.28,
    "B2":85.0,
    "B3":88.35,
    "TN1":85.0,
    "B5":85.0
}

def simulate_battery_values_revised(
    current_simulation_step,
    cumulative_ah_discharged_start_of_step,
    capacity,
    charged_ah_feature
):
    # 1. Capacity Degradation
    degradation_multiplier = 1.0 - (DEGRADATION_RATE_PER_INTERVAL * current_simulation_step)
    current_effective_capacity = capacity * max(0.20, degradation_multiplier)

    # 2. Simulate Current Draw (using consistent 5-minute interval logic)
    intervals_per_24h = (24 * 60) / SIM_INTERVAL_MINUTES
    time_of_day_cycle_position = (current_simulation_step % intervals_per_24h) / intervals_per_24h

    # Sinusoidal current pattern with some noise
    base_current_value = 8 + 7 * np.sin(2 * np.pi * time_of_day_cycle_position) # e.g. 8A avg, 15A peak, 1A min
    simulated_current = min(MAX_SIMULATED_CURRENT, base_current_value + np.random.normal(0, 0.3)) # Reduced noise
    simulated_current = max(0.1, simulated_current) # Ensure current is always discharging a bit

    # 3. Amp-hours for this interval
    ah_discharged_this_interval = simulated_current * SIM_INTERVAL_HOURS

    # 4. Update Cumulative Amp-hours Discharged
    total_cumulative_ah_discharged = cumulative_ah_discharged_start_of_step + ah_discharged_this_interval

    # 5. Calculate Remaining Capacity
    if current_effective_capacity > 0:
        current_rc = 1.0 - (total_cumulative_ah_discharged / current_effective_capacity)
    else:
        current_rc = 0.0
    current_rc = max(0.0, min(1.0, current_rc))

    # 6. Calculate Voltage
    simulated_voltage = MIN_OPERATIONAL_VOLTAGE + (MAX_OPERATIONAL_VOLTAGE - MIN_OPERATIONAL_VOLTAGE) * current_rc
    simulated_voltage += np.random.normal(0, 0.03)
    simulated_voltage = max(MIN_OPERATIONAL_VOLTAGE - 0.2, simulated_voltage)
    simulated_voltage = min(MAX_OPERATIONAL_VOLTAGE + 0.2, simulated_voltage)

    # 7. Calculate Power
    simulated_power = simulated_voltage * simulated_current

    # 8. Remaining Capacity Percentage
    remaining_capacity_percent = current_rc * 100.0

    return (
        simulated_current,
        simulated_voltage,
        ah_discharged_this_interval,
        total_cumulative_ah_discharged,
        simulated_power,
        remaining_capacity_percent,
        charged_ah_feature
    )


In [6]:
def simulator_predictor_manipulator(n):
    TIME_STEP = 0
    cumulative_ah_total = 0.0

    data_buffer = []

    battery_keys_list = list(battery_type_capacities.keys())
    current_battery_code = battery_keys_list[n]
    # battery_type_index = n # if you need to store the index itself
    capacity = float(battery_type_capacities[current_battery_code])

    charged_ah = capacity


    print(f"Starting simulation for: {current_battery_code} (Nominal Capacity: {capacity} Ah)")

    for i in range(1000):
        TIME_STEP += 1

        current, voltage, ah_out, updated_cumulative_ah, power, remaining_perc, charged_ah_val = \
            simulate_battery_values_revised(
                current_simulation_step=(TIME_STEP - 1),
                cumulative_ah_discharged_start_of_step=cumulative_ah_total,
                capacity=capacity,
                charged_ah_feature=charged_ah
            )

        cumulative_ah_total = updated_cumulative_ah

        if voltage < MIN_OPERATIONAL_VOLTAGE:
            print(f"INFO: Voltage ({voltage:.2f}V) fell below cutoff ({MIN_OPERATIONAL_VOLTAGE:.2f}V) at step {TIME_STEP}. Stopping.")
            break
        if remaining_perc < 0.1:
            print(f"INFO: Remaining capacity ({remaining_perc:.2f}%) is critical at step {TIME_STEP}. Stopping.")
            break

        base_row = {
            'Current': float(current),
            'Voltage': float(voltage),
            'Ah Out': float(ah_out),
            'Cumulative Actual Disch Ah': float(cumulative_ah_total),
            'Power': float(power),
            'Remaining Capacity': float(remaining_perc),
            'type': current_battery_code,
            'capacity': float(capacity),
            'charged': float(charged_ah_val),
        }

        data_buffer.append(base_row)

        if len(data_buffer) >= TIME_STEPS:
            X_input_df = pd.DataFrame(data_buffer)

            pred_model1 = model1.predict(X_input_df)
            X_input_df_for_model2 = X_input_df.copy()
            X_input_df_for_model2['prediction'] = pred_model1[-1]

            y_pred_model2 = model2.predict(X_input_df_for_model2).tolist()
            y_pred_model2.append(None)
            for i in data2:
                data2.loc[len(data2)] = y_pred_model2

            data_buffer.pop(0)

In [7]:
t1 = threading.Thread(target=simulator_predictor_manipulator, args=(0,))
t2 = threading.Thread(target=simulator_predictor_manipulator, args=(1,))
t3 = threading.Thread(target=simulator_predictor_manipulator, args=(2,))
t4 = threading.Thread(target=simulator_predictor_manipulator, args=(3,))
t5 = threading.Thread(target=simulator_predictor_manipulator, args=(4,))


t1.start()
t2.start()
t3.start()
t4.start()
t5.start()

t1.join()
t2.join()
t3.join()
t4.join()
t5.join()

Starting simulation for: B1 (Nominal Capacity: 81.28 Ah)
Starting simulation for: B2 (Nominal Capacity: 85.0 Ah)
Starting simulation for: B3 (Nominal Capacity: 88.35 Ah)
Starting simulation for: TN1 (Nominal Capacity: 85.0 Ah)
Starting simulation for: B5 (Nominal Capacity: 85.0 Ah)
INFO: Remaining capacity (0.00%) is critical at step 78. Stopping.
INFO: Voltage (9.34V) fell below cutoff (9.40V) at step 80. Stopping.
INFO: Voltage (9.38V) fell below cutoff (9.40V) at step 81. Stopping.
INFO: Voltage (9.38V) fell below cutoff (9.40V) at step 81. Stopping.
INFO: Remaining capacity (0.00%) is critical at step 84. Stopping.


In [8]:
data2

,pred1,pred2,pred3,pred4,pred5,pred6,pred7,pred8,pred9,pred10,MEAN
0,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,NaN
1,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,NaN
2,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,NaN
3,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,NaN
4,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2032,5955.239848,5949.601143,5956.944762,5950.366957,5952.209142,5947.140232,5947.140232,5946.656652,5941.587742,5941.587742,NaN
2033,5976.145282,5972.597300,5971.088405,5963.376625,5963.376625,5962.928730,5967.758831,5962.242026,5957.376240,5962.688487,NaN
2034,5846.802873,5850.598405,5847.340136,5847.650024,5847.122242,5845.000280,5845.000280,5841.089760,5837.567134,5827.232981,NaN
2036,5760.270920,5972.597300,5971.088405,5761.790302,5756.145714,5962.928730,5967.758831,5962.242026,5957.376240,5962.688487,NaN


In [14]:
for index, row in data2.iterrows():
    pred_values = [row[f'pred{i}'] for i in range(1, 11)]
    data2.at[index, 'MEAN'] = np.mean(pred_values)


In [15]:
data2

,pred1,pred2,pred3,pred4,pred5,pred6,pred7,pred8,pred9,pred10,MEAN
0,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,21491.917862
1,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,21491.917862
2,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,21491.917862
3,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,21491.917862
4,21492.398078,21496.906817,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21492.398078,21494.621228,21480.864025,21491.917862
...,...,...,...,...,...,...,...,...,...,...,...
2032,5955.239848,5949.601143,5956.944762,5950.366957,5952.209142,5947.140232,5947.140232,5946.656652,5941.587742,5941.587742,5948.847445
2033,5976.145282,5972.597300,5971.088405,5963.376625,5963.376625,5962.928730,5967.758831,5962.242026,5957.376240,5962.688487,5965.957855
2034,5846.802873,5850.598405,5847.340136,5847.650024,5847.122242,5845.000280,5845.000280,5841.089760,5837.567134,5827.232981,5843.540411
2036,5760.270920,5972.597300,5971.088405,5761.790302,5756.145714,5962.928730,5967.758831,5962.242026,5957.376240,5962.688487,5903.488696
